# CHD-VLM Evaluation — HuggingFace Models

**Notebook 1 of 3** — Run open-weight HF models on Colab GPU.

**Models**: MedGemma-4B, MedGemma-27B, LLaVA-Med-7B, Qwen2.5-VL-7B  
**Prompts**: ZSD, RCE, CoT, RAC  
**Dataset**: CHD-CXR (828 pediatric chest X-rays, 4 classes)  

**GPU requirements**:
- T4 (16 GB): MedGemma-4B, LLaVA-Med-7B (4-bit), Qwen2.5-VL-7B (4-bit)
- A100 (40 GB): MedGemma-27B (4-bit) + all T4 models

After this notebook, run `02_api_eval.ipynb` for API models, then `03_analysis.ipynb`.

## 0. Environment Setup

In [ ]:
# Verify GPU
!nvidia-smi
import torch
print(f"CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    print(f"VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")

In [ ]:
# Clone repo and install dependencies
import os

REPO_URL = "https://github.com/rushankgoyal/chd-eval.git"
REPO_DIR = "/content/chd-eval"

if not os.path.exists(REPO_DIR):
    !git clone {REPO_URL} {REPO_DIR}
else:
    !cd {REPO_DIR} && git pull

%cd {REPO_DIR}/new

!pip install -q -r requirements.txt
print("Installation complete.")

## 1. Authentication

In [ ]:
from google.colab import userdata
import os

# HuggingFace token (required for gated MedGemma models)
os.environ["HF_TOKEN"] = userdata.get("HF_TOKEN")

from huggingface_hub import whoami
info = whoami(token=os.environ["HF_TOKEN"])
print(f"Logged in as: {info['name']}")

## 2. Dataset Loading

In [ ]:
# Option A: Mount Google Drive (recommended if dataset is large)
# from google.colab import drive
# drive.mount('/content/drive')
# DATA_ROOT = "/content/drive/MyDrive/chd-cxr-dataset"

# Option B: Upload and unzip directly
# from google.colab import files
# uploaded = files.upload()  # upload chd-cxr.zip
# !unzip -q chd-cxr.zip -d /content/data
# DATA_ROOT = "/content/data/chd-cxr"

# Set your dataset path here:
DATA_ROOT = "/content/data/chd-cxr"  # <-- EDIT THIS

import sys
sys.path.insert(0, "/content/chd-eval/new")

from src.data.dataset import load_samples_from_directory, LABELS

samples = load_samples_from_directory(DATA_ROOT)
print(f"\nTotal samples: {len(samples)}")
for label in LABELS:
    count = sum(s.label == label for s in samples)
    print(f"  {label}: {count}")

## 3. Configuration

In [ ]:
from src.evaluation.runner import ExperimentConfig

# Choose experiment config:
#   pilot.yaml     — 2 models × 2 prompts × 20 samples  (smoke test, ~5 min)
#   hf_full.yaml   — 4 HF models × 4 prompts × all samples  (full run)

EXPERIMENT = "configs/experiments/hf_full.yaml"  # <-- EDIT THIS

# To skip MedGemma-27B (requires A100), comment it out in hf_full.yaml

config = ExperimentConfig.from_yaml(EXPERIMENT)
print(f"Models   : {[m.display_name for m in config.model_configs]}")
print(f"Prompts  : {config.prompt_ids}")
print(f"Conditions: {len(config.model_configs) * len(config.prompt_ids)}")
print(f"Max samples: {config.max_samples or len(samples)}")
print(f"Results dir: {config.results_dir}")

## 4. Evaluation Loop

In [ ]:
import logging
logging.basicConfig(level=logging.INFO, format="%(levelname)s  %(message)s")

from src.evaluation.runner import EvaluationRunner

runner = EvaluationRunner(config, samples)
df = runner.run()

print(f"\nResults shape: {df.shape}")
print(f"Expected rows: {len(samples if config.max_samples is None else samples[:config.max_samples])} × {len(config.model_configs)} × {len(config.prompt_ids)}")
df.head()

In [ ]:
# Parse rate summary
parse_summary = (
    df.groupby(["model_name", "prompt_id"])["parse_success"]
    .agg(["mean", "sum", "count"])
    .rename(columns={"mean": "parse_rate", "sum": "n_parsed", "count": "n_total"})
    .reset_index()
)
parse_summary["parse_rate"] = parse_summary["parse_rate"].map("{:.1%}".format)
print(parse_summary.to_string(index=False))

In [ ]:
# Download combined HF results CSV
from google.colab import files
import shutil, os

results_path = str(config.results_dir / "combined.csv")
print(f"Downloading: {results_path}")
files.download(results_path)